# RHCR Baseline Window Sweep

在本 notebook 中直接设置多组 `simulation_window` / `planning_window`，仅运行 baseline（`--use_learned_cost=false`），结果输出到 `RHCR/exp2`。

In [1]:
from pathlib import Path
import csv
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def parse_completed_tasks(tasks_file: Path) -> int:
    if not tasks_file.exists():
        return -1
    lines = tasks_file.read_text(encoding='utf-8', errors='ignore').splitlines()
    total = 0
    for ln in lines[1:]:
        for seg in [s for s in ln.split(';') if s.strip()]:
            parts = [p for p in seg.split(',') if p != '']
            if len(parts) == 3:
                try:
                    t = int(parts[1])
                except ValueError:
                    continue
                if t >= 0:
                    total += 1
    return total

def run_one(task):
    sim_w = task['sim_w']
    plan_w = task['plan_w']
    k = task['k']
    seed = task['seed']
    cmd = task['cmd']
    run_dir = task['run_dir']
    rhcr_root = task['rhcr_root']
    simulation_time = task['simulation_time']

    status = 'ok'
    with (run_dir / 'run.log').open('w', encoding='utf-8') as logf:
        p = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, cwd=str(rhcr_root))
    rc = p.returncode
    if rc != 0:
        status = 'failed'

    completed = parse_completed_tasks(run_dir / 'tasks.txt')
    throughput = (completed / float(simulation_time)) if completed >= 0 else -1.0
    row = [sim_w, plan_w, k, seed, status, rc, completed, throughput, str(run_dir)]
    print(f'[{status}] sim={sim_w} plan={plan_w} k={k} seed={seed} completed={completed} throughput={throughput:.4f}')
    return row


In [2]:
# ===== 配置区域（直接改这里）=====
repo_root = Path('/home/shiqi/masterarbeit')
rhcr_root = repo_root / 'RHCR'
lifelong_bin = rhcr_root / 'lifelong'
map_path = rhcr_root / 'maps' / 'Paris_1_256c4.map'

scenario = 'KIVA'
solver = 'PBS'   # PBS / ECBS / WHCA / LRA
suboptimal_bound = 1.5  # 仅 ECBS 用
dummy_paths = False
simulation_time = 256

agents = [ 64, 128, 256, 384, 512]
seeds = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19 ]

# 重点：要比较的窗口组合
window_pairs = [
    (1, 5),
    (1, 10),
    (1, 20),
    (5, 5),
    (5, 10),
    (5, 20),
]

num_process = 6  # 1=串行, >1=并行

# 输出根目录（按你的要求放 exp2）
output_root = rhcr_root / 'exp2'
run_name = time.strftime('rhcr_baseline_windows_%Y%m%d_%H%M%S')
run_root = output_root / run_name
run_root.mkdir(parents=True, exist_ok=True)
summary_csv = run_root / 'summary.csv'

print('run_root =', run_root)
print('lifelong =', lifelong_bin)
print('map =', map_path)
assert lifelong_bin.exists(), f'Missing binary: {lifelong_bin}'
assert map_path.exists(), f'Missing map: {map_path}'


run_root = /home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseline_windows_20260514_153151
lifelong = /home/shiqi/masterarbeit/RHCR/lifelong
map = /home/shiqi/masterarbeit/RHCR/maps/Paris_1_256c4.map


In [ ]:
rows = []
with summary_csv.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow([
        'simulation_window', 'planning_window', 'num_agents', 'seed',
        'status', 'returncode', 'completed_tasks', 'throughput_per_step', 'run_dir'
    ])

tasks = []
for sim_w, plan_w in window_pairs:
    for k in agents:
        for seed in seeds:
            run_dir = run_root / f'sim_{sim_w}_plan_{plan_w}' / f'agents_{k}' / f'seed_{seed}'
            run_dir.mkdir(parents=True, exist_ok=True)

            cmd = [
                str(lifelong_bin),
                '-m', str(map_path),
                '--scenario', str(scenario),
                '-k', str(k),
                '--simulation_window', str(sim_w),
                '--planning_window', str(plan_w),
                '--solver', str(solver),
                '--seed', str(seed),
                '--simulation_time', str(simulation_time),
                '--dummy_paths', 'true' if dummy_paths else 'false',
                '--use_learned_cost', 'false',
                '-o', str(run_dir),
            ]
            if str(solver).upper() == 'ECBS':
                cmd.extend(['--suboptimal_bound', str(float(suboptimal_bound))])

            (run_dir / 'command.sh').write_text(' '.join(cmd) + '\n', encoding='utf-8')
            tasks.append({
                'sim_w': sim_w,
                'plan_w': plan_w,
                'k': k,
                'seed': seed,
                'cmd': cmd,
                'run_dir': run_dir,
                'rhcr_root': rhcr_root,
                'simulation_time': simulation_time,
            })

print(f'Launching {len(tasks)} runs with num_process={num_process}')
if int(num_process) <= 1:
    for task in tasks:
        rows.append(run_one(task))
else:
    with ThreadPoolExecutor(max_workers=int(num_process)) as ex:
        futures = [ex.submit(run_one, task) for task in tasks]
        for fut in as_completed(futures):
            rows.append(fut.result())

rows.sort(key=lambda r: (int(r[0]), int(r[1]), int(r[2]), int(r[3])))
with summary_csv.open('a', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    for row in rows:
        w.writerow(row)

print('Done. summary =', summary_csv)


Launching 300 runs with num_process=6
[ok] sim=1 plan=5 k=64 seed=11 completed=309 throughput=1.2070
[ok] sim=1 plan=5 k=64 seed=14 completed=313 throughput=1.2227
[ok] sim=1 plan=5 k=64 seed=15 completed=306 throughput=1.1953
[ok] sim=1 plan=5 k=64 seed=13 completed=319 throughput=1.2461
[ok] sim=1 plan=5 k=64 seed=12 completed=327 throughput=1.2773
[ok] sim=1 plan=5 k=64 seed=10 completed=306 throughput=1.1953
[ok] sim=1 plan=5 k=64 seed=17 completed=312 throughput=1.2188
[ok] sim=1 plan=5 k=64 seed=16 completed=329 throughput=1.2852
[ok] sim=1 plan=5 k=64 seed=19 completed=306 throughput=1.1953
[ok] sim=1 plan=5 k=64 seed=18 completed=317 throughput=1.2383
[ok] sim=1 plan=5 k=128 seed=10 completed=605 throughput=2.3633
[ok] sim=1 plan=5 k=128 seed=11 completed=614 throughput=2.3984
[ok] sim=1 plan=5 k=128 seed=15 completed=631 throughput=2.4648
[ok] sim=1 plan=5 k=128 seed=13 completed=641 throughput=2.5039
[ok] sim=1 plan=5 k=128 seed=14 completed=612 throughput=2.3906
[ok] sim=1 p

In [ ]:
import pandas as pd

df = pd.read_csv(summary_csv)
display(df.head(20))

ok = df[(df['status'] == 'ok') & (df['throughput_per_step'] >= 0)].copy()
if len(ok) > 0:
    agg = (ok.groupby(['simulation_window', 'planning_window', 'num_agents'])['throughput_per_step']
             .agg(['mean', 'std', 'count'])
             .reset_index()
             .sort_values(['simulation_window', 'planning_window', 'num_agents']))
    display(agg)
else:
    print('No successful runs.')


,simulation_window,planning_window,num_agents,seed,status,returncode,completed_tasks,throughput_per_step,run_dir
0,1,5,32,10,ok,0,320,1.250000,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
1,1,5,32,11,ok,0,305,1.191406,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
2,1,5,32,12,ok,0,312,1.218750,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
3,1,5,32,13,ok,0,329,1.285156,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
4,1,5,32,14,ok,0,325,1.269531,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
5,1,5,32,15,ok,0,331,1.292969,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
6,1,5,32,16,ok,0,314,1.226562,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
7,1,5,32,17,ok,0,298,1.164062,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
8,1,5,32,18,ok,0,316,1.234375,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...
9,1,5,32,19,ok,0,312,1.218750,/home/shiqi/masterarbeit/RHCR/exp2/rhcr_baseli...


,simulation_window,planning_window,num_agents,mean,std,count
0,1,5,32,1.235156,0.040587,10
1,1,5,64,2.262109,0.039039,10
2,1,5,96,2.853906,0.193737,10
3,1,5,128,2.492578,0.331796,10
4,1,5,160,1.780469,0.249313,10
5,1,5,192,1.451172,0.266328,10
6,1,10,32,1.236328,0.033413,10
7,1,10,64,2.303906,0.043685,10
8,1,10,96,3.061719,0.091772,10
9,1,10,128,3.464062,0.280706,10
